In [ ]:
%pip install -q psycopg2-binary sqlalchemy pandas xgboost shap matplotlib


In [ ]:
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text


# XGBoost duration modeling

Fetch task-run data, explore volume–duration correlations, train XGBoost regressors, and evaluate clustering features.

Outputs (gitignored): `data/xgboost_tasks_data.csv`, `data/xgboost_fetch_checkpoints/`.
Requires `DATABASE_URL`.


In [ ]:
import os
import random
import time
from datetime import datetime, timedelta, timezone

from sqlalchemy import create_engine, text

DATABASE_URL = os.environ.get("DATABASE_URL")
if not DATABASE_URL:
    raise RuntimeError("Set DATABASE_URL (see .env.example)")
OUTPUT_CSV = DATA_DIR / "xgboost_tasks_data.csv"
ROOT = Path.cwd()
if not (ROOT / "data" / "case_studies").is_dir() and (ROOT.parent / "data" / "case_studies").is_dir():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = DATA_DIR / "xgboost_fetch_checkpoints"  # delete dir to refetch after query changes

# Smaller windows + checkpoints help on read replicas.
LOOKBACK_DAYS = 30
CHUNK_DAYS = 1
MAX_RETRIES = 8

QUERY = text("""
SELECT
    -- ── Task run identity ─────────────────────────────────────────────────────
    t.app_id,
    t.task_name,
    t.task_id,
    t.app_pit,
    t.start_time,
    t.end_time,
    t.status,

    -- ── Duration ──────────────────────────────────────────────────────────────
    te.task__duration                                                       AS task_duration_seconds,
    te.executors__run_time__used                                            AS executors_run_time_seconds,
    te.executors__tasks_duration__used                                      AS executors_tasks_duration_seconds,

    -- ── CALCULATE_USD_COST inputs: VCore-seconds ('VCore') ───────────────────
    -- These are the raw resources_value when impact_unit = 'VCore'
    te.task__vcore_time__used                                               AS vcore_seconds_used,
    te.task__vcore_time__allocated                                          AS vcore_seconds_allocated,
    te.executors__vcore_time__used                                          AS executors_vcore_seconds_used,
    te.executors__vcore_time__allocated                                     AS executors_vcore_seconds_allocated,
    te.executors__cpu_time__used                                            AS executors_cpu_time_seconds,
    te.executors__used_vcore_time_of_retried_tasks                          AS vcore_seconds_wasted_on_retries,

    -- ── CALCULATE_USD_COST inputs: GB-seconds ('GB') ─────────────────────────
    -- These are the raw resources_value when impact_unit = 'GB'
    te.task__memory_time__allocated                                         AS memory_gb_seconds_allocated,
    te.executors__memory_time__used                                         AS memory_gb_seconds_used,

    -- ── Memory GB (instantaneous allocations & peak usage) ───────────────────
    te.executor__memory__allocated                                          AS executor_total_memory_gb,
    te.executor__memory_heap__allocated                                     AS executor_heap_memory_allocated_gb,
    te.executor__memory_heap__max_used                                      AS executor_heap_memory_peak_gb,
    te.executor__memory_off_heap__allocated                                 AS executor_off_heap_memory_allocated_gb,
    te.executor__memory_off_heap__max_used                                  AS executor_off_heap_memory_peak_gb,
    te.driver__memory__allocated                                            AS driver_total_memory_allocated_gb,
    te.driver__memory__max_used                                             AS driver_memory_peak_gb,
    te.driver__heap_memory__allocated                                       AS driver_heap_memory_allocated_gb,
    te.driver__heap_memory__max_used                                        AS driver_heap_memory_peak_gb,

    -- ── IO / data volume ──────────────────────────────────────────────────────
    te.task__total_io_bytes                                                 AS total_io_bytes,
    te.task__disk_bytes_spilled                                             AS disk_bytes_spilled,
    -- Spill ratio: what fraction of IO was caused by spill (read+write = 2x spilled)
    CASE
        WHEN te.task__total_io_bytes > 0
        THEN 2.0 * te.task__disk_bytes_spilled / te.task__total_io_bytes
        ELSE 0
    END                                                                     AS spill_ratio,

    -- ── IO records (from metrics; not in task_enrichments) ────────────────────
    rec.input_records_read,
    rec.output_records_written,
    rec.shuffle_records_read,
    rec.shuffle_records_written,

    -- ── GC ────────────────────────────────────────────────────────────────────
    te.executors__jvm_gc_time                                               AS executors_jvm_gc_seconds,

    -- ── Spark config params (configured values per run) ───────────────────────
    te.task__executors_dynamic_allocation_min_executors__param              AS spark_dynamic_alloc_min_executors,
    te.task__executors_dynamic_allocation_max_executors__param              AS spark_dynamic_alloc_max_executors,
    te.task__executor_instances__param                                      AS spark_executor_instances,
    te.executor__memory_heap__allocated                                     AS spark_executor_memory_gb,
    te.executor__memory_off_heap__allocated                                 AS spark_executor_memory_overhead_gb,
    te.driver__heap_memory__allocated                                       AS spark_driver_memory_gb,
    te.driver__cores                                                        AS spark_driver_cores,
    te.task__shuffle_partitions__param                                      AS spark_shuffle_partitions,
    te.task__task_cpus__param                                               AS spark_task_cpus,
    te.executor__cores                                                      AS spark_executor_cores,
    te.task__dynamic_is_allocation_enabled__param                           AS is_dynamic_allocation,

    -- ── Cluster / cloud config ────────────────────────────────────────────────
    te.cluster_min_workers,
    te.cluster_max_workers,
    te.cluster_workers,
    te.workers_availability                                                 AS aws_availability,
    te.worker_type                                                          AS worker_instance_type,
    te.driver_type                                                          AS driver_instance_type,
    te.cloud_provider,
    -- hidden vCores forced by machine memory-to-vcore ratio (see CALCULATE_UNUSED_VCORE_TIME)
    GREATEST(
        (te.executor__memory__allocated / NULLIF(ts.machine_memory_to_vcore_ratio, 0) - te.executor__cores)
        * te.task__duration, 0
    )                                                                       AS executors_vcores_unused,
    (
        (
            te.task__vcore_time__allocated
            + GREATEST(
                (te.executor__memory__allocated / NULLIF(ts.machine_memory_to_vcore_ratio, 0) - te.executor__cores)
                * te.task__duration, 0
              )
        ) * ts.vcore_price / 3600
        + te.task__memory_time__allocated * ts.memory_price / 3600
    )                                                                       AS run_cost_usd
FROM tasks AS t
JOIN task_enrichments AS te
    ON te.task_id = t.task_id
JOIN tenant_settings AS ts
    ON ts.tenant_id = t.tenant_id
LEFT JOIN (
    SELECT
        m.task_id,
        MAX(m.metric_value) FILTER (WHERE mc.metric_type = 'input_records_read')       AS input_records_read,
        MAX(m.metric_value) FILTER (WHERE mc.metric_type = 'output_records_written')   AS output_records_written,
        MAX(m.metric_value) FILTER (WHERE mc.metric_type = 'shuffle_records_read')     AS shuffle_records_read,
        MAX(m.metric_value) FILTER (WHERE mc.metric_type = 'shuffle_records_written')  AS shuffle_records_written
    FROM metrics AS m
    JOIN metrics_conf AS mc USING (metric_id)
    JOIN tasks AS t_rec ON t_rec.task_id = m.task_id
    WHERE t_rec.start_time >= :start_ts
      AND t_rec.start_time < :end_ts
    GROUP BY m.task_id
) AS rec
    ON rec.task_id = t.task_id
WHERE t.start_time >= :start_ts
  AND t.start_time < :end_ts
""")

RETRYABLE_MARKERS = (
    "SerializationFailure",
    "conflict with recovery",
    "SSL connection has been closed",
    "connection already closed",
    "server closed the connection",
)


def is_retryable(exc: Exception) -> bool:
    msg = f"{type(exc).__name__}: {getattr(exc, 'orig', exc)}"
    return any(marker in msg for marker in RETRYABLE_MARKERS)


def checkpoint_path(start_ts: datetime, end_ts: datetime) -> Path:
    return CHECKPOINT_DIR / f"{start_ts.date()}_{end_ts.date()}.csv"


def fetch_chunk(start_ts: datetime, end_ts: datetime) -> pd.DataFrame:
    params = {"start_ts": start_ts, "end_ts": end_ts}
    for attempt in range(1, MAX_RETRIES + 1):
        engine = create_engine(
            DATABASE_URL,
            pool_pre_ping=True,
            pool_size=1,
            max_overflow=0,
        )
        try:
            with engine.connect() as conn:
                conn.execute(text("SET statement_timeout = '900s'"))
                return pd.read_sql(QUERY, conn, params=params)
        except Exception as exc:
            if not is_retryable(exc) or attempt == MAX_RETRIES:
                raise
            wait_s = min(90, 2**attempt + random.uniform(0, 2))
            print(
                f"  retry {attempt}/{MAX_RETRIES} in {wait_s:.0f}s "
                f"({type(exc).__name__})"
            )
            time.sleep(wait_s)
        finally:
            engine.dispose()
    raise RuntimeError("unreachable")


CHECKPOINT_DIR.mkdir(exist_ok=True)
range_end = datetime.now(timezone.utc)
range_start = range_end - timedelta(days=LOOKBACK_DAYS)

chunks: list[pd.DataFrame] = []
cursor = range_start
while cursor < range_end:
    chunk_end = min(cursor + timedelta(days=CHUNK_DAYS), range_end)
    cp = checkpoint_path(cursor, chunk_end)

    if cp.exists():
        chunk = pd.read_csv(cp, low_memory=False)
        print(f"{cursor.date()} → {chunk_end.date()}: {len(chunk):,} rows (checkpoint)")
    else:
        chunk = fetch_chunk(cursor, chunk_end)
        chunk.to_csv(cp, index=False)
        print(f"{cursor.date()} → {chunk_end.date()}: {len(chunk):,} rows (fetched)")

    chunks.append(chunk)
    cursor = chunk_end

xgboost_tasks_data = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()
xgboost_tasks_data.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(xgboost_tasks_data):,} rows to {OUTPUT_CSV.resolve()}")


In [ ]:
xgboost_tasks_data = xgboost_tasks_data[xgboost_tasks_data.status == 'COMPLETED']

In [ ]:
xgboost_tasks_data.columns

## Volume vs duration correlation

Pearson and Spearman correlation between task-run **volume** metrics and `task_duration_seconds` (COMPLETED runs only). Metrics with no data in the current CSV are skipped.

In [ ]:
VOLUME_COLUMNS = [
    "vcore_seconds_used",
    "memory_gb_seconds_used",
    "executors_vcore_seconds_used",
    "total_io_bytes",
    "disk_bytes_spilled",
    "spill_ratio",
    "input_records_read",
    "output_records_written",
    "shuffle_records_read",
    "shuffle_records_written",
]
y = "task_duration_seconds"

volume_cols = [c for c in VOLUME_COLUMNS if c in xgboost_tasks_data.columns]
missing_volume_cols = [c for c in VOLUME_COLUMNS if c not in xgboost_tasks_data.columns]

rows = []
for col in volume_cols:
    sub = (
        xgboost_tasks_data[[col, y]]
        .apply(pd.to_numeric, errors="coerce")
        .dropna()
    )
    sub = sub[(sub[col] > 0) & (sub[y] > 0)]
    if len(sub) < 3:
        continue
    rows.append(
        {
            "volume_metric": col,
            "n": len(sub),
            "pearson_r": sub[col].corr(sub[y]),
            "spearman_r": sub[col].corr(sub[y], method="spearman"),
        }
    )

volume_duration_corr = (
    pd.DataFrame(rows)
    .sort_values("pearson_r", key=lambda s: s.abs(), ascending=False)
    .reset_index(drop=True)
)

print(f"Volume metrics in dataset: {len(volume_cols)} / {len(VOLUME_COLUMNS)}")
if missing_volume_cols:
    print("Missing from CSV (refetch to include):", ", ".join(missing_volume_cols))

display(
    volume_duration_corr.style.format(
        {
            "n": "{:,.0f}",
            "pearson_r": "{:,.3f}",
            "spearman_r": "{:,.3f}",
        }
    )
)


In [ ]:
X = ['executor_total_memory_gb',
       'executor_heap_memory_allocated_gb', 'executor_heap_memory_peak_gb',
       'executor_off_heap_memory_allocated_gb',
       'executor_off_heap_memory_peak_gb', 'driver_total_memory_allocated_gb',
       'driver_memory_peak_gb', 'driver_heap_memory_allocated_gb',
       'driver_heap_memory_peak_gb', 'total_io_bytes', 'disk_bytes_spilled',
       'spill_ratio', 
       'spark_dynamic_alloc_min_executors',
       'spark_dynamic_alloc_max_executors', 'spark_executor_instances',
       'spark_executor_memory_gb', 'spark_executor_memory_overhead_gb',
       'spark_driver_memory_gb', 'spark_driver_cores',
       'spark_shuffle_partitions', 'spark_task_cpus', 'spark_executor_cores',
       'is_dynamic_allocation', 'cluster_min_workers', 'cluster_max_workers',
       'cluster_workers', 'aws_availability', 'worker_instance_type',
       'driver_instance_type', 'cloud_provider', 'executors_vcores_unused']
y = 'task_duration_seconds'

In [ ]:
import matplotlib.pyplot as plt

# Pearson correlation: each feature in X vs y
corr_df = xgboost_tasks_data[X + [y]].copy()

# Label-encode categoricals so every column is numeric for corr()
cat_cols = corr_df.select_dtypes(include=["object", "bool", "category"]).columns
for col in cat_cols:
    corr_df[col] = pd.factorize(corr_df[col].astype(str))[0]

corr_df = corr_df.apply(pd.to_numeric, errors="coerce")

corr_with_y = (
    corr_df[X]
    .corrwith(corr_df[y])
    .sort_values(key=lambda s: s.abs(), ascending=False)
    .rename("corr_with_task_duration_seconds")
)

print("Correlation of each X feature with y (Pearson):")
display(corr_with_y.to_frame())

# Full correlation matrix: all X + y
corr_matrix = corr_df[X + [y]].corr()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

corr_with_y.plot(kind="barh", ax=axes[0], color="steelblue")
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set_xlabel("Pearson correlation")
axes[0].set_title(f"Each X feature vs {y}")
axes[0].invert_yaxis()

im = axes[1].imshow(
    corr_matrix.values,
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    aspect="auto",
)
axes[1].set_xticks(range(len(corr_matrix.columns)))
axes[1].set_yticks(range(len(corr_matrix.columns)))
axes[1].set_xticklabels(corr_matrix.columns, rotation=90, ha="right")
axes[1].set_yticklabels(corr_matrix.columns)
axes[1].set_title(f"Correlation matrix: X + {y}")
fig.colorbar(im, ax=axes[1], label="correlation")
plt.tight_layout()
plt.show()

## XGBoost regression (baseline)

Default train/test split, raw target scale. See **improved model** section below for log target, group split, and task history.

In [ ]:
%pip install -q xgboost shap
# macOS: if XGBoost fails to load, run in terminal once: brew install libomp

import numpy as np
import matplotlib.pyplot as plt
import shap
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

# --- Prepare features (uses X and y from previous cell) ---
model_df = xgboost_tasks_data[X + [y]].copy()

cat_cols = model_df[X].select_dtypes(include=["object", "bool", "string"]).columns.tolist()
if cat_cols:
    model_df = pd.get_dummies(model_df, columns=cat_cols, dummy_na=True)

for col in model_df.columns:
    if col != y:
        model_df[col] = pd.to_numeric(model_df[col], errors="coerce")
model_df[y] = pd.to_numeric(model_df[y], errors="coerce")
model_df = model_df.dropna(subset=[y])

feature_cols = [c for c in model_df.columns if c != y]
X_mat = model_df[feature_cols]
y_vec = model_df[y]

X_train, X_test, y_train, y_test = train_test_split(
    X_mat, y_vec, test_size=0.2, random_state=42
)
print(f"train: {len(X_train):,} rows | test: {len(X_test):,} rows | features: {len(feature_cols)}")

# --- Default XGBoost regressor ---
model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\nTest set evaluation:")
print(f"  MAE:  {mae:,.2f}")
print(f"  RMSE: {rmse:,.2f}")
print(f"  R²:   {r2:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(y_test, y_pred, alpha=0.15, s=8)
max_val = max(y_test.max(), y_pred.max())
axes[0].plot([0, max_val], [0, max_val], "r--", lw=1)
axes[0].set_xlabel(f"Actual {y}")
axes[0].set_ylabel(f"Predicted {y}")
axes[0].set_title("Actual vs predicted (test)")

residuals = y_test - y_pred
axes[1].hist(residuals, bins=50, edgecolor="black", alpha=0.75)
axes[1].set_xlabel("Residual")
axes[1].set_ylabel("Count")
axes[1].set_title("Residual distribution (test)")
plt.tight_layout()
plt.show()

# --- SHAP (sample test rows for speed) ---
SHAP_SAMPLE = min(2000, len(X_test))
X_shap = X_test.sample(SHAP_SAMPLE, random_state=42)

explainer = shap.TreeExplainer(model)
shap_values = explainer(X_shap)

print(f"\nSHAP summary ({SHAP_SAMPLE:,} test rows):")
shap.summary_plot(shap_values, X_shap, show=False)
plt.tight_layout()
plt.show()

shap.summary_plot(shap_values, X_shap, plot_type="bar", show=False)
plt.tight_layout()
plt.show()

## Improved model: log target, history, evaluation modes

**Why the first "improved" run looked worse (R² ≈ 0.06):** group split by `(app_id, task_name)` puts **only brand-new tasks** in test. Config-only features barely predict those cold starts. The baseline (random split, R² ≈ 0.74) reused the same tasks in train and test — optimistic for deployment.

This section compares:
1. **Repeat-task (random split)** — realistic when you've seen `(app_id, task_name)` before (production default)
2. **Cold-task (group split)** — stress test for never-seen tasks
3. **Cold-task, repeat runs only** — test rows where `hist_run_count ≥ 1`

Uses per-task chronology history + **app-level stats fit on train only** (helps cold tasks in known apps).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import shap
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from xgboost import XGBRegressor

TASK_KEY = ["app_id", "task_name"]
HIST_FEATURES = [
    "hist_run_count",
    "hist_median_duration",
    "hist_p90_duration",
    "hist_mean_duration",
]
APP_HIST_FEATURES = [
    "app_hist_median",
    "app_hist_mean",
    "app_hist_p90",
    "app_hist_run_count",
]


def eval_metrics(y_true, y_pred, label: str) -> dict:
    y_pred = np.clip(y_pred, 0, None)
    return {
        "mode": label,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "n": len(y_true),
    }


def build_base_df() -> pd.DataFrame:
    """Config X + per-task chronology history (past runs only)."""
    history_df = xgboost_tasks_data[TASK_KEY + [y, "start_time"]].copy()
    history_df["start_time"] = pd.to_datetime(
        history_df["start_time"], format="ISO8601", utc=True
    )
    history_df[y] = pd.to_numeric(history_df[y], errors="coerce")
    history_df = history_df.dropna(subset=[y]).sort_values("start_time")

    g = history_df.groupby(TASK_KEY, group_keys=False)
    history_df["hist_run_count"] = g.cumcount()
    history_df["hist_median_duration"] = g[y].transform(
        lambda s: s.shift(1).expanding().median()
    )
    history_df["hist_p90_duration"] = g[y].transform(
        lambda s: s.shift(1).expanding().quantile(0.9)
    )
    history_df["hist_mean_duration"] = g[y].transform(
        lambda s: s.shift(1).expanding().mean()
    )

    global_fill = {
        "hist_median_duration": history_df[y].median(),
        "hist_p90_duration": history_df[y].quantile(0.9),
        "hist_mean_duration": history_df[y].mean(),
    }
    for col, val in global_fill.items():
        history_df[col] = history_df[col].fillna(val)
    history_df["hist_run_count"] = history_df["hist_run_count"].fillna(0)

    base = xgboost_tasks_data[X + TASK_KEY + [y]].copy()
    base[HIST_FEATURES] = history_df[HIST_FEATURES].sort_index()
    return base


def add_app_history_from_train(df: pd.DataFrame, train_idx, global_fallback: float) -> pd.DataFrame:
    """App-level duration stats from train runs only (no test labels)."""
    train = df.iloc[train_idx]
    app_stats = (
        train.groupby("app_id")[y]
        .agg(
            app_hist_median="median",
            app_hist_mean="mean",
            app_hist_p90=lambda s: s.quantile(0.9),
            app_hist_run_count="count",
        )
        .reset_index()
    )
    out = df.merge(app_stats, on="app_id", how="left")
    for col in ["app_hist_median", "app_hist_mean", "app_hist_p90"]:
        out[col] = out[col].fillna(global_fallback)
    out["app_hist_run_count"] = out["app_hist_run_count"].fillna(0)
    return out


def prepare_matrix(df: pd.DataFrame):
    work = df.copy()
    feature_base = X + HIST_FEATURES + APP_HIST_FEATURES
    cat_cols = work[feature_base].select_dtypes(
        include=["object", "bool", "string"]
    ).columns.tolist()
    if cat_cols:
        work = pd.get_dummies(work, columns=cat_cols, dummy_na=True)

    for col in work.columns:
        if col not in TASK_KEY + [y, "app_id"]:
            work[col] = pd.to_numeric(work[col], errors="coerce")
    work[y] = pd.to_numeric(work[y], errors="coerce")

    feature_cols = [c for c in work.columns if c not in TASK_KEY + [y, "app_id"]]
    return work, feature_cols


def train_xgb(X_train, y_train, X_test, y_test):
    y_train = pd.to_numeric(y_train, errors="coerce").to_numpy()
    y_test = pd.to_numeric(y_test, errors="coerce").to_numpy()
    y_train_log = np.log1p(y_train)
    y_test_log = np.log1p(y_test)
    assert np.isfinite(y_train_log).all() and np.isfinite(y_test_log).all()
    model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=2000,
        learning_rate=0.03,
        max_depth=8,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        early_stopping_rounds=50,
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X_train, y_train_log, eval_set=[(X_test, y_test_log)], verbose=False)
    return model, np.clip(np.expm1(model.predict(X_test)), 0, None)


def split_indices(df: pd.DataFrame, split_mode: str):
    n = len(df)
    if split_mode == "repeat_task":
        return train_test_split(np.arange(n), test_size=0.2, random_state=42)
    groups = df["app_id"].astype(str) + "||" + df["task_name"].astype(str)
    return next(
        GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(
            df, groups=groups
        )
    )


def build_split_df(base_df: pd.DataFrame, split_mode: str):
    """App history from a train fold on base_df, then clean rows and re-split."""
    train_base, _ = split_indices(base_df, split_mode)
    global_fallback = base_df.iloc[train_base][y].median()
    df = add_app_history_from_train(base_df, train_base, global_fallback)
    df, feature_cols = prepare_matrix(df)

    df[feature_cols] = (
        df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    )
    y_vals = pd.to_numeric(df[y], errors="coerce").to_numpy()
    valid = np.isfinite(y_vals) & (y_vals >= 0)
    dropped = (~valid).sum()
    if dropped:
        print(f"  dropped {dropped:,} rows with invalid {y}")
    df = df.loc[valid].reset_index(drop=True)

    train_idx, test_idx = split_indices(df, split_mode)
    return df, feature_cols, train_idx, test_idx


base_df = build_base_df()
results = []

for split_mode in ("repeat_task", "cold_task"):
    label = (
        "Repeat-task (random split)"
        if split_mode == "repeat_task"
        else "Cold-task (group split)"
    )
    df, feature_cols, train_idx, test_idx = build_split_df(base_df, split_mode)

    X_train = df[feature_cols].iloc[train_idx]
    X_test = df[feature_cols].iloc[test_idx]
    y_train = df[y].iloc[train_idx]
    y_test = df[y].iloc[test_idx]
    hist_run_count = df["hist_run_count"]

    model, y_pred = train_xgb(X_train, y_train, X_test, y_test)
    results.append(eval_metrics(y_test, y_pred, label))

    if split_mode == "cold_task":
        repeat_mask = hist_run_count.iloc[test_idx].to_numpy() >= 1
        if repeat_mask.any():
            results.append(
                eval_metrics(
                    y_test.to_numpy()[repeat_mask],
                    y_pred[repeat_mask],
                    label + " | hist_run_count>=1",
                )
            )

# Production-like model for plots (repeat-task)
df, feature_cols, train_idx, test_idx = build_split_df(base_df, "repeat_task")
X_train = df[feature_cols].iloc[train_idx]
X_test = df[feature_cols].iloc[test_idx]
y_train = df[y].iloc[train_idx]
y_test = df[y].iloc[test_idx]
model_improved, y_pred = train_xgb(X_train, y_train, X_test, y_test)

summary = pd.DataFrame(results).set_index("mode")
print("Comparison (test set, seconds scale):")
display(summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(y_test, y_pred, alpha=0.15, s=8)
mx = max(y_test.max(), y_pred.max())
axes[0].plot([0, mx], [0, mx], "r--", lw=1)
axes[0].set_xlabel(f"Actual {y}")
axes[0].set_ylabel(f"Predicted {y}")
axes[0].set_title("Repeat-task split (production-like)")

axes[1].hist(y_test - y_pred, bins=50, edgecolor="black", alpha=0.75)
axes[1].set_xlabel("Residual (seconds)")
axes[1].set_title("Residuals")
plt.tight_layout()
plt.show()

SHAP_SAMPLE = min(2000, len(X_test))
X_shap = X_test.sample(SHAP_SAMPLE, random_state=42)
shap_values = shap.TreeExplainer(model_improved)(X_shap)
shap.summary_plot(shap_values, X_shap, show=False)
plt.tight_layout()
plt.show()


In [ ]:
y_pred

In [ ]:
y_test

## Task similarity: PCA + KMeans (`sklearn.pipeline.Pipeline`)

Built-in sklearn chain: **StandardScaler → PCA (2D) → KMeans**. Clusters are defined in PCA space; the scatter plot matches what KMeans saw.


In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

TASK_KEY = ["app_id", "task_name"]
MIN_RUNS_PER_TASK = 6
N_PCA_COMPONENTS = 2   # 2D plot; clustering runs in this PCA space
N_CLUSTERS = 8         # starting hint; auto-tuned below

CAT_IN_X = {
    "is_dynamic_allocation",
    "aws_availability",
    "worker_instance_type",
    "driver_instance_type",
    "cloud_provider",
}
NUMERIC_X = [c for c in X if c not in CAT_IN_X]


def make_pca_kmeans_pipeline(n_clusters: int) -> Pipeline:
    """StandardScaler → PCA → KMeans (sklearn Pipeline)."""
    return Pipeline(
        steps=[
            ("scale", StandardScaler()),
            ("pca", PCA(n_components=N_PCA_COMPONENTS, random_state=42)),
            (
                "kmeans",
                KMeans(n_clusters=n_clusters, random_state=42, n_init=10),
            ),
        ]
    )


# --- Build task profiles (one row per app_id + task_name) ---
profile_df = xgboost_tasks_data[TASK_KEY + ["task_id"] + NUMERIC_X + [y]].copy()
for col in NUMERIC_X + [y]:
    profile_df[col] = pd.to_numeric(profile_df[col], errors="coerce")

agg_spec = {col: ["mean", "median"] for col in NUMERIC_X + [y]}
agg_spec["task_id"] = "nunique"
task_profiles = profile_df.groupby(TASK_KEY, as_index=False).agg(agg_spec)
task_profiles.columns = [
    "_".join(c).strip("_") if isinstance(c, tuple) else c
    for c in task_profiles.columns
]
task_profiles = task_profiles.rename(columns={"task_id_nunique": "run_count"})
task_profiles = task_profiles[task_profiles["run_count"] >= MIN_RUNS_PER_TASK].reset_index(
    drop=True
)
print(f"{len(task_profiles):,} tasks with >={MIN_RUNS_PER_TASK} runs")

feature_cols = [
    c for c in task_profiles.columns if c.endswith("_mean") or c.endswith("_median")
]
X_profile = task_profiles[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

# --- Pick k by silhouette in PCA space ---
best_k, best_sil = N_CLUSTERS, -1.0
for k in range(4, min(16, len(task_profiles) // 20)):
    pipe_k = make_pca_kmeans_pipeline(k)
    labels_k = pipe_k.fit_predict(X_profile)
    X_pca_k = pipe_k.named_steps["pca"].transform(
        pipe_k.named_steps["scale"].transform(X_profile)
    )
    sil = silhouette_score(X_pca_k, labels_k)
    if sil > best_sil:
        best_k, best_sil = k, sil

print(f"Auto-selected k={best_k} (silhouette in PCA space={best_sil:.3f})")

# --- Fit final PCA + KMeans pipeline ---
cluster_pipe = make_pca_kmeans_pipeline(best_k)
task_profiles["cluster"] = cluster_pipe.fit_predict(X_profile)

X_pca = cluster_pipe.named_steps["pca"].transform(
    cluster_pipe.named_steps["scale"].transform(X_profile)
)
pca = cluster_pipe.named_steps["pca"]
task_profiles["pca1"] = X_pca[:, 0]
task_profiles["pca2"] = X_pca[:, 1]

print(
    f"PCA explained variance: {pca.explained_variance_ratio_.sum():.1%} "
    f"(PC1 {pca.explained_variance_ratio_[0]:.1%}, "
    f"PC2 {pca.explained_variance_ratio_[1]:.1%})"
)

# --- 2D scatter (same space KMeans clustered in) ---
fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(
    task_profiles["pca1"],
    task_profiles["pca2"],
    c=task_profiles["cluster"],
    cmap="tab10",
    alpha=0.6,
    s=20 + 2 * np.sqrt(task_profiles["run_count"]),
    edgecolors="none",
)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
ax.set_title(f"PCA + KMeans pipeline (n={len(task_profiles):,} tasks, k={best_k})")
plt.colorbar(scatter, ax=ax, label="cluster")
plt.tight_layout()
plt.show()

loadings = pd.DataFrame(
    pca.components_.T,
    index=feature_cols,
    columns=["PC1_loading", "PC2_loading"],
)
print("Top PC1 drivers:")
display(loadings["PC1_loading"].abs().sort_values(ascending=False).head(8).to_frame())

# --- Cluster summaries (original feature space for interpretability) ---
cluster_summaries = []
for cid in sorted(task_profiles["cluster"].unique()):
    mask = task_profiles["cluster"] == cid
    cluster_summaries.append(
        {
            "cluster": cid,
            "n_tasks": mask.sum(),
            "median_runs": task_profiles.loc[mask, "run_count"].median(),
            "median_duration_s": task_profiles.loc[mask, f"{y}_median"].median(),
            "mean_pca1": task_profiles.loc[mask, "pca1"].mean(),
            "mean_pca2": task_profiles.loc[mask, "pca2"].mean(),
        }
    )
print("\nCluster summaries:")
display(pd.DataFrame(cluster_summaries).set_index("cluster"))




def cluster_cohesion_metrics(
    X_pca: np.ndarray,
    labels: np.ndarray,
    X_features: np.ndarray | None = None,
) -> pd.DataFrame:
    """How tight / similar is each cluster internally?"""
    from sklearn.metrics import silhouette_samples

    sil_all = (
        silhouette_samples(X_pca, labels)
        if len(np.unique(labels)) > 1 and len(labels) > len(np.unique(labels))
        else np.full(len(labels), np.nan)
    )

    rows = []
    for cid in sorted(np.unique(labels)):
        mask = labels == cid
        idx = np.where(mask)[0]
        pts = X_pca[idx]
        n = len(idx)
        centroid = pts.mean(axis=0)
        dists = np.linalg.norm(pts - centroid, axis=1)

        if n >= 2:
            sim = cosine_similarity(pts)
            pair_cos = sim[np.triu_indices(n, k=1)]
            mean_cos = pair_cos.mean()
            median_cos = np.median(pair_cos)
            min_cos = pair_cos.min()
            wcss = float((dists**2).sum())
        else:
            mean_cos = median_cos = min_cos = np.nan
            wcss = 0.0

        row = {
            "cluster": cid,
            "n_tasks": n,
            "mean_dist_to_centroid_pca": dists.mean(),
            "std_dist_to_centroid_pca": dists.std(ddof=1) if n > 1 else 0.0,
            "max_dist_to_centroid_pca": dists.max(),
            "wcss_pca": wcss,
            "mean_pairwise_cosine": mean_cos,
            "median_pairwise_cosine": median_cos,
            "min_pairwise_cosine": min_cos,
            "mean_silhouette": sil_all[mask].mean(),
        }
        if X_features is not None and n > 1:
            row["mean_feature_variance"] = X_features[idx].var(axis=0).mean()
        rows.append(row)

    return pd.DataFrame(rows).set_index("cluster")


labels = task_profiles["cluster"].to_numpy()
cohesion = cluster_cohesion_metrics(
    X_pca,
    labels,
    X_features=StandardScaler().fit_transform(X_profile),
)
if f"{y}_median" in task_profiles.columns:
    cohesion["median_duration_iqr"] = task_profiles.groupby("cluster")[f"{y}_median"].apply(
        lambda s: s.quantile(0.75) - s.quantile(0.25)
    )

print("\nWithin-cluster cohesion (lower distance / higher cosine = more similar tasks):")
display(cohesion.round(4))

print(
    "\nHow to read:\n"
    "  • mean_pairwise_cosine → 1.0 = identical profiles, ~0 = unrelated (in PCA space)\n"
    "  • mean_dist_to_centroid_pca → avg spread around cluster center (lower = tighter)\n"
    "  • wcss_pca → total squared spread (lower = tighter; compare clusters with similar n)\n"
    "  • mean_silhouette → [-1, 1]; higher = better separated AND internally cohesive\n"
    "  • mean_feature_variance → avg variance of raw profile features inside the cluster\n"
    "  • median_duration_iqr → spread of typical run duration within the cluster (seconds)"
)

def top_similar_pairs(cluster_id: int, top_n: int = 5) -> pd.DataFrame:
    """Cosine similarity in PCA space (same space as KMeans)."""
    mask = task_profiles["cluster"] == cluster_id
    idx = np.where(mask)[0]
    if len(idx) < 2:
        return pd.DataFrame()
    sim = cosine_similarity(X_pca[idx])
    pairs = []
    for i in range(len(idx)):
        for j in range(i + 1, len(idx)):
            pairs.append((idx[i], idx[j], sim[i, j]))
    pairs.sort(key=lambda t: t[2], reverse=True)
    return pd.DataFrame(
        [
            {
                "similarity": score,
                "app_id_a": task_profiles.iloc[i]["app_id"],
                "task_a": task_profiles.iloc[i]["task_name"],
                "app_id_b": task_profiles.iloc[j]["app_id"],
                "task_b": task_profiles.iloc[j]["task_name"],
            }
            for i, j, score in pairs[:top_n]
        ]
    )


for cid in sorted(task_profiles["cluster"].unique()):
    pairs = top_similar_pairs(cid)
    if pairs.empty:
        continue
    print(f"\nCluster {cid} — most similar pairs (cosine in PCA space):")
    display(pairs)


def representative_tasks(cluster_id: int, top_n: int = 5) -> pd.DataFrame:
    mask = task_profiles["cluster"] == cluster_id
    idx = np.where(mask)[0]
    centroid = X_pca[idx].mean(axis=0)
    dist = np.linalg.norm(X_pca[idx] - centroid, axis=1)
    rep_idx = idx[np.argsort(dist)[:top_n]]
    return task_profiles.iloc[rep_idx][
        TASK_KEY + ["run_count", f"{y}_median", "pca1", "pca2", "cluster"]
    ]


print("\nRepresentative (closest to PCA centroid) tasks per cluster:")
for cid in sorted(task_profiles["cluster"].unique()):
    print(f"\n--- Cluster {cid} ---")
    display(representative_tasks(cid))


In [ ]:
task_profiles

## Does cluster help XGBoost?

Same pipeline as the improved model, with and without **cluster_id** (from PCA+KMeans) as an extra categorical feature. Compares on repeat-task and cold-task splits.

In [ ]:
from xgboost import XGBRegressor


def prepare_matrix_cluster(df: pd.DataFrame, include_cluster: bool):
    work = df.copy()
    feature_base = X + HIST_FEATURES + APP_HIST_FEATURES
    extra_cat: list[str] = []
    if include_cluster:
        work["cluster_id"] = work["cluster"].fillna(-1).astype(int).astype(str)
        extra_cat = ["cluster_id"]

    cat_cols = (
        work[feature_base].select_dtypes(include=["object", "bool", "string"]).columns.tolist()
        + extra_cat
    )
    if cat_cols:
        work = pd.get_dummies(work, columns=cat_cols, dummy_na=True)

    for col in work.columns:
        if col not in TASK_KEY + [y, "app_id", "cluster", "cluster_id"]:
            work[col] = pd.to_numeric(work[col], errors="coerce")
    work[y] = pd.to_numeric(work[y], errors="coerce")
    drop_cols = TASK_KEY + [y, "app_id", "cluster", "cluster_id"]
    feature_cols = [c for c in work.columns if c not in drop_cols]
    return work, feature_cols


def build_split_df_cluster(base_df: pd.DataFrame, split_mode: str, include_cluster: bool):
    train_base, _ = split_indices(base_df, split_mode)
    global_fallback = base_df.iloc[train_base][y].median()
    df = add_app_history_from_train(base_df, train_base, global_fallback)
    df, feature_cols = prepare_matrix_cluster(df, include_cluster=include_cluster)

    df[feature_cols] = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    y_vals = pd.to_numeric(df[y], errors="coerce").to_numpy()
    valid = np.isfinite(y_vals) & (y_vals >= 0)
    if (~valid).sum():
        print(f"  dropped {(~valid).sum():,} rows with invalid {y}")
    df = df.loc[valid].reset_index(drop=True)

    train_idx, test_idx = split_indices(df, split_mode)
    return df, feature_cols, train_idx, test_idx


# Attach cluster label to each run (from task_profiles cell)
base_df_cluster = build_base_df()
cluster_map = task_profiles[TASK_KEY + ["cluster"]]
base_df_cluster = base_df_cluster.merge(cluster_map, on=TASK_KEY, how="left")
print(
    f"Runs with cluster label: {base_df_cluster['cluster'].notna().sum():,} / "
    f"{len(base_df_cluster):,} "
    f"({base_df_cluster['cluster'].notna().mean():.1%})"
)

cluster_results = []
for split_mode in ("repeat_task", "cold_task"):
    split_label = (
        "repeat_task" if split_mode == "repeat_task" else "cold_task"
    )
    for include_cluster in (False, True):
        tag = "with_cluster" if include_cluster else "no_cluster"
        df, feature_cols, train_idx, test_idx = build_split_df_cluster(
            base_df_cluster, split_mode, include_cluster=include_cluster
        )
        X_train = df[feature_cols].iloc[train_idx]
        X_test = df[feature_cols].iloc[test_idx]
        y_train = df[y].iloc[train_idx]
        y_test = df[y].iloc[test_idx]

        _, y_pred = train_xgb(X_train, y_train, X_test, y_test)
        m = eval_metrics(y_test, y_pred, f"{split_label}_{tag}")
        m["split"] = split_label
        m["cluster_feature"] = include_cluster
        m["n_features"] = len(feature_cols)
        cluster_results.append(m)

cluster_cmp = pd.DataFrame(cluster_results)
display(cluster_cmp)

pivot = cluster_cmp.pivot_table(
    index="split",
    columns="cluster_feature",
    values=["MAE", "RMSE", "R2"],
)
print("\nSide-by-side (False = no cluster, True = with cluster):")
display(pivot.round(4))

for split_label in pivot.index:
    try:
        dr2 = pivot.loc[split_label, ("R2", True)] - pivot.loc[split_label, ("R2", False)]
        dmae = pivot.loc[split_label, ("MAE", True)] - pivot.loc[split_label, ("MAE", False)]
        print(
            f"{split_label}: ΔR²={dr2:+.4f}, ΔMAE={dmae:+.1f}s "
            f"({'cluster helps' if dr2 > 0 and dmae < 0 else 'cluster does not help' if dr2 <= 0 else 'mixed'})"
        )
    except KeyError:
        pass

# Cluster dummy importances (repeat-task + cluster model)
df, feature_cols, train_idx, test_idx = build_split_df_cluster(
    base_df_cluster, "repeat_task", include_cluster=True
)
X_train = df[feature_cols].iloc[train_idx]
X_test = df[feature_cols].iloc[test_idx]
model_c, _ = train_xgb(X_train, df[y].iloc[train_idx], X_test, df[y].iloc[test_idx])
cluster_cols = [c for c in feature_cols if c.startswith("cluster_id_")]
if cluster_cols:
    imp = pd.Series(model_c.feature_importances_, index=feature_cols)
    print("\nCluster feature importances (repeat-task model):")
    display(imp[cluster_cols].sort_values(ascending=False).to_frame("importance"))
    print(f"Share of total importance: {imp[cluster_cols].sum() / imp.sum():.2%}")